# Imports

In [ ]:
import matplotlib 
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

The project CSV files are stored in the data/processed

In [3]:
ad_revenue = pd.read_csv('../data/processed/ad_revenue.csv')
iap_products = pd.read_csv('../data/processed/iap_products.csv')
matches = pd.read_csv('../data/processed/matches.csv')
purchases = pd.read_csv('../data/processed/purchases.csv')
sessions = pd.read_csv('../data/processed/sessions.csv')
ua_spend = pd.read_csv('../data/processed/ua_spend.csv')
users = pd.read_csv('../data/processed/users.csv')

# Retention curves by channel

In [ ]:
EXTRACT = pd.Timestamp('2024-07-22')
sessions['fecha'] = pd.to_datetime(sessions['fecha'])
users['fecha_instalacion'] = pd.to_datetime(users['fecha_instalacion'])


sj = sessions.merge(users[['user_id', 'canal_adquisicion', 'fecha_instalacion']], on ='user_id', how='inner')
sj['dd'] = (sj['fecha'] - sj['fecha_instalacion']).dt.days

filas = []
for N in (1, 7, 30):
    # N days: 
    coh = users[(EXTRACT - users['fecha_instalacion']).dt.days >= N] 
    cohc = coh.groupby('canal_adquisicion')['user_id'].nunique()
    ret = sj[sj['dd'] == N].groupby('canal_adquisicion')['user_id'].nunique()
    r = (ret / cohc * 100).round(1)
    for channel in ['coinrush', 'meta', 'google', 'apple_search', 'tiktok', 'organico']: 
        filas.append({'day': N, 'channel':channel, 'retention': r.get(channel,0)})
    print(f"D{N}: coinrush {r['coinrush']}% | meta {r['meta']}% | apple_search {r['apple_search']}%")
    
curve = pd.DataFrame(filas)
plt.figure(figsize=(8,5))
for _c in curve['channel'].unique(): 
    _d = curve[curve['channel'] == _c]
    plt.plot(_d['day'], _d['retention'], marker='o', label=_c)
plt.legend()
plt.title('Retention by channel (D1, D7, D3)')
plt.show()
plt.savefig('../outputs/retention_by_channel.png')

D1: coinrush 37.0% | meta 43.4% | apple_search 47.2%
D7: coinrush 2.4% | meta 22.5% | apple_search 25.4%
D30: coinrush 0.8% | meta 9.8% | apple_search 13.1%


C:\Users\asus\AppData\Local\Temp\ipykernel_9244\3883202196.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# payer conversion by channel

In [48]:
users['fecha_instalacion'] = pd.to_datetime(users['fecha_instalacion'])
sales = purchases[(purchases['reembolso'] == 0) & (purchases['moneda'] == 'EUR')].copy()
sales = sales[sales['importe'] > 0]

inst = users.groupby('canal_adquisicion')['user_id'].nunique()
pay = sales.merge(users[['user_id','canal_adquisicion']], on='user_id', how='inner')

# conversion by channel  = unique payers / installs by channel 
conv  = (pay.groupby('canal_adquisicion')['user_id'].nunique() / inst * 100).round(2).sort_values()

for channel, v in conv.items():
    print(f'{channel}: {v}%')
    
plt.figure(figsize=(8, 5))
plt.barh(conv.index, conv.values, color='lightblue')
plt.title('Payer conversion by channel (%)')
plt.show()
plt.savefig('../outputs/payer_conversion_by_channel.png')

coinrush: 0.11%
tiktok: 1.0%
google: 1.79%
organico: 2.0%
meta: 2.31%
apple_search: 2.56%


C:\Users\asus\AppData\Local\Temp\ipykernel_9244\3549609225.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ROAS BY CHANNEL 

In [78]:
users['fecha_instalacion'] = pd.to_datetime(users['fecha_instalacion'])
sales = purchases[(purchases['reembolso'] == 0) & (purchases['moneda'] == 'EUR')].copy()
sales = sales[sales['importe'] > 0]

inst = users.groupby('canal_adquisicion')['user_id'].nunique()
spend = ua_spend.groupby('canal')['gasto'].sum()
cpi = spend / inst
pay = sales.merge(users[['user_id','canal_adquisicion']], on='user_id', how ='inner')
iap_net = pay.groupby('canal_adquisicion')['importe'].sum() * 0.70
adsj = ad_revenue.merge(users[['user_id','canal_adquisicion']], on='user_id', how='inner')
net =iap_net.add(adsj.groupby('canal_adquisicion')['ingreso'].sum(), fill_value=0)
ltv = net/inst

# ROAS = LTV / CPI 
j_pay = cpi[cpi > 0].index
roas = (ltv[j_pay] / cpi[j_pay]).round(2).sort_values()

for channel, v in roas.items():
    state = 'LOST' if v < 1 else 'WIN'
    print(f'{channel}: ROAS {v} ({state})')
    
plt.figure(figsize=(8, 5))
plt.bar(roas.index, roas.values, color='#a3e635')
plt.axhline(1, color='#22d3ee', linestyle='--')   # linea de rentabilidad
plt.title('ROAS by channel (break-even line = 1)')
plt.show()
plt.savefig('../outputs/roas_by_channel.png')

coinrush: ROAS 0.27 (LOST)
tiktok: ROAS 1.06 (WIN)
google: ROAS 1.16 (WIN)
meta: ROAS 1.35 (WIN)
apple_search: ROAS 1.42 (WIN)


C:\Users\asus\AppData\Local\Temp\ipykernel_9244\2526605781.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ROAS figures

In [90]:
users['fecha_instalacion'] = pd.to_datetime(users['fecha_instalacion'])
sales = purchases[(purchases['reembolso'] == 0) & (purchases['moneda'] == 'EUR')].copy()
sales = sales[sales['importe'] > 0]

inst = users.groupby('canal_adquisicion')['user_id'].nunique()
cpi = ua_spend.groupby('canal')['gasto'].sum() / inst
pay = sales.merge(users[['user_id','canal_adquisicion']], on='user_id', how ='inner')
iap_net = pay.groupby('canal_adquisicion')['importe'].sum() * 0.70
adsj = ad_revenue.merge(users[['user_id','canal_adquisicion']], on='user_id', how='inner')
net =iap_net.add(adsj.groupby('canal_adquisicion')['ingreso'].sum(), fill_value=0)
ltv = net/inst

roas_coinrush = round( ltv['coinrush'] / cpi['coinrush'],2)
getback_coinrush = round(roas_coinrush * 1000)

print(f"CoinRush net LTV: {ltv['coinrush']:.2f} EUR")
print(f"CoinRush CPI: {cpi['coinrush']:.2f} EUR")
print(f"CoinRush ROAS: {roas_coinrush:.2f}")
print(f"BIG NUMBER: for every 1000 EUR spent on CoinRush, we get back {getback_coinrush} EUR")

CoinRush net LTV: 0.12 EUR
CoinRush CPI: 0.45 EUR
CoinRush ROAS: 0.27
BIG NUMBER: for every 1000 EUR spent on CoinRush, we get back 270 EUR
